# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression and Random Forest, compared against a hand rule.**

My lane ranks pages by decline risk so a reviewer can work down a queue. That makes
this a binary classification problem used as a ranking: the model outputs a
probability, and the probability is the priority score.

Logistic Regression is the simple, readable option — its coefficients say which
direction each feature pushes. Random Forest is the stronger option, able to learn
interactions a linear model cannot.

I am not using clustering. Clustering describes groups without a target, and I have
a defined target (`is_declining`). I am not using Gradient Boosting either: with only
21 features and a known-imperfect proxy label, the extra tuning surface would buy
accuracy I could not defend.

In [13]:
%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected")

Connected


In [14]:
DECISION_DATE = "2026-02-28"

features = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clicks_feb,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_visible_feb,
               SUM(CASE WHEN report_date <= DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h1,
               SUM(CASE WHEN report_date >  DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h2,
               MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS has_ga4
        FROM {FEB}
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_mar
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_feb,
        f.clicks_feb,
        f.days_visible_feb,
        ROUND(f.clicks_feb::FLOAT / NULLIF(f.imp_feb, 0), 5) AS ctr_feb,
        ROUND(f.imp_feb::FLOAT / NULLIF(f.days_visible_feb, 0), 2) AS imp_per_active_day,
        ROUND(f.imp_h2::FLOAT / NULLIF(f.imp_h1, 0), 3) AS trend_within_feb,
        f.has_ga4,
        COALESCE(d.word_count, 0) AS word_count,
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
        COALESCE(d.search_volume, 0) AS search_volume,
        COALESCE(d.competition, 0) AS competition,
        COALESCE(d.backlinks, 0) AS backlinks,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        COALESCE(d.content_type, 'unknown') AS content_type,
        COALESCE(d.competition_level, 'unknown') AS competition_level,
        COALESCE(d.main_intent, 'unknown') AS main_intent,
        COALESCE(m.imp_mar, 0) AS imp_mar
    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE f.imp_feb >= 50
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

print(f"Rows: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 93,654


,client_hash_id,content_hash_id,imp_feb,clicks_feb,days_visible_feb,ctr_feb,imp_per_active_day,trend_within_feb,has_ga4,word_count,word_count_missing,search_volume,competition,backlinks,content_age_days,content_type,competition_level,main_intent,imp_mar
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,246.0,1.0,28,0.00407,8.79,0.662,0,3168,0,0,0.00,0,144,keyword article,LOW,informational,331.0
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,137.0,0.0,27,0.00000,5.07,1.076,0,4135,0,20,0.03,9,144,keyword article,LOW,commercial,33.0
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,121.0,0.0,28,0.00000,4.32,0.833,0,3211,0,0,0.00,0,144,keyword article,LOW,informational,145.0
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,174.0,0.0,12,0.00000,14.50,NaN,0,3465,0,0,0.00,0,144,keyword article,LOW,informational,461.0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,164.0,0.0,28,0.00000,5.86,0.451,0,3149,0,0,0.00,0,144,keyword article,LOW,informational,232.0


In [15]:
df = features.copy()

FEB_DAYS, MAR_DAYS = 28, 31
imp_rate_feb = df["imp_feb"] / FEB_DAYS
imp_rate_mar = df["imp_mar"] / MAR_DAYS
df["is_declining"] = (imp_rate_mar < 0.8 * imp_rate_feb).astype(int)

df["no_h1_impressions"] = df["trend_within_feb"].isna().astype(int)
df["trend_within_feb"] = df["trend_within_feb"].fillna(1.0)
df["content_age_days"] = df["content_age_days"].fillna(-1)

print(f"Rows: {len(df):,}")
print(f"Positive rate: {df['is_declining'].mean():.1%} ({df['is_declining'].sum():,} declining)")

Rows: 93,654
Positive rate: 27.6% (25,887 declining)


In [16]:
encoded_prefixes = ("ctype_", "intent_")
df = df.drop(columns=[c for c in df.columns if c.startswith(encoded_prefixes)], errors="ignore")
df = df.drop(columns=["competition_level_ord"], errors="ignore")

competition_order = {"unknown": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3}
df["competition_level_ord"] = df["competition_level"].map(competition_order).fillna(0).astype(int)

one_hot = pd.get_dummies(df[["content_type", "main_intent"]],
                         prefix=["ctype", "intent"], dtype=int)
df = pd.concat([df, one_hot], axis=1)

drop_cols = ["client_hash_id", "content_hash_id", "imp_mar", "is_declining",
             "content_type", "competition_level", "main_intent"]
model_features = [c for c in df.columns if c not in drop_cols]

print(f"Model-ready features: {len(model_features)}")

Model-ready features: 23


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped by client, so whole clients stay out of training.**

The 93,654 pages belong to 55 clients, and pages from one client share a site, a
topic mix, and a traffic pattern. A random row split would let the model see some of
a client's pages in training and be tested on the rest — it could then recognise
clients rather than learn why pages decline.

I use `GroupKFold` on `client_hash_id` with 5 folds. Every page of a client sits
entirely in train or entirely in test.

Client sizes are very uneven, so a single grouped split would be unstable: whether a
large client lands in train or test would move the score. Reporting the mean and
spread across 5 folds is the honest version of a number that a single split would
make look more precise than it is.

In [17]:
from sklearn.model_selection import GroupKFold

groups = df["client_hash_id"]
gkf = GroupKFold(n_splits=5)

print(f"Clients: {groups.nunique()}  Pages: {len(df):,}")
for i, (tr, te) in enumerate(gkf.split(df, df["is_declining"], groups), 1):
    print(f"  fold {i}: train {len(tr):>6,} | test {len(te):>6,} | "
          f"test clients {groups.iloc[te].nunique():>2} | "
          f"test positive rate {df['is_declining'].iloc[te].mean():.1%}")

Clients: 38  Pages: 93,654
  fold 1: train 73,296 | test 20,358 | test clients  1 | test positive rate 24.3%
  fold 2: train 75,334 | test 18,320 | test clients  8 | test positive rate 32.8%
  fold 3: train 75,318 | test 18,336 | test clients  8 | test positive rate 18.8%
  fold 4: train 75,334 | test 18,320 | test clients 11 | test positive rate 42.6%
  fold 5: train 75,334 | test 18,320 | test clients 10 | test positive rate 20.0%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
import numpy as np

# Week-4 style hand rule, retargeted at decline so the comparison is on the same label.
# The w04 rule ranked by missed clicks, a different question; this one ranks decline risk
# using the same three-signal logic: momentum, visibility, age.
def decline_baseline_score(frame):
    momentum_risk = (1 - frame["trend_within_feb"]).clip(lower=0, upper=1)
    visibility_risk = 1 - (frame["days_visible_feb"] / 28)
    age_risk = (frame["content_age_days"].clip(lower=0, upper=365) / 365)
    return (0.60 * momentum_risk + 0.25 * visibility_risk + 0.15 * age_risk).round(4)


df["baseline_score"] = decline_baseline_score(df)
print(df["baseline_score"].describe().round(3).to_string())

count    93654.000
mean         0.178
std          0.141
min          0.012
25%          0.084
50%          0.147
75%          0.212
max          0.965


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k=50):
    top = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top].mean())


X = df[model_features]
y = df["is_declining"]

models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1),
}

results = {name: {"auc": [], "ap": [], "p50": []} for name in models}
results["Baseline rule"] = {"auc": [], "ap": [], "p50": []}

for tr, te in gkf.split(X, y, groups):
    y_te = y.iloc[te]

    base_scores = df["baseline_score"].iloc[te].to_numpy()
    results["Baseline rule"]["auc"].append(roc_auc_score(y_te, base_scores))
    results["Baseline rule"]["ap"].append(average_precision_score(y_te, base_scores))
    results["Baseline rule"]["p50"].append(precision_at_k(y_te, base_scores))

    for name, model in models.items():
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        results[name]["auc"].append(roc_auc_score(y_te, p))
        results[name]["ap"].append(average_precision_score(y_te, p))
        results[name]["p50"].append(precision_at_k(y_te, p))

comparison = pd.DataFrame({
    name: {
        "ROC AUC": f"{np.mean(m['auc']):.3f} +/- {np.std(m['auc']):.3f}",
        "Avg precision": f"{np.mean(m['ap']):.3f} +/- {np.std(m['ap']):.3f}",
        "Precision@50": f"{np.mean(m['p50']):.3f} +/- {np.std(m['p50']):.3f}",
    }
    for name, m in results.items()
}).T

print(f"Base rate (always-declining guess): {y.mean():.3f}\n")
print(comparison.to_string())

Base rate (always-declining guess): 0.276

                             ROC AUC    Avg precision     Precision@50
Logistic Regression  0.592 +/- 0.055  0.373 +/- 0.132  0.648 +/- 0.185
Random Forest        0.685 +/- 0.045  0.486 +/- 0.161  0.836 +/- 0.142
Baseline rule        0.633 +/- 0.085  0.444 +/- 0.189  0.680 +/- 0.228


**Model vs baseline, 5-fold GroupKFold on client_hash_id.**

| | ROC AUC | Avg precision | Precision@50 |
|---|---|---|---|
| Baseline rule | 0.633 +/- 0.085 | 0.444 +/- 0.189 | 0.680 +/- 0.228 |
| Logistic Regression | 0.592 +/- 0.055 | 0.373 +/- 0.132 | 0.648 +/- 0.185 |
| Random Forest | 0.685 +/- 0.045 | 0.486 +/- 0.161 | 0.836 +/- 0.142 |

Base rate: 0.276. Every score above is measured on the same folds, the same
population, and the same label.

**The linear model loses to the hand rule.** Logistic Regression scores 0.592 ROC AUC
against the rule's 0.633. This is a result, not a failure: it says the relationship
between my features and decline is not linear, and that the rule's three signals —
momentum, visibility, age — carry more together than a linear combination of 21
features does. Complexity did not help here; the model class mattered.

**Random Forest is ahead, but the margin is not as clean as the means suggest.**
Precision@50 rises from 0.680 to 0.836, and ROC AUC from 0.633 to 0.685. But the
spreads overlap: the rule ranges roughly 0.45 to 0.91 across folds, the forest 0.69
to 0.98. In some folds the rule may have matched or beaten the model. The directional
read is that the forest ranks better; I cannot claim it wins on every client mix.

**The spread is the point, not noise.** Client sizes are very uneven, so which clients
land in the test fold changes the score substantially. A single grouped split would
have reported one number and hidden this. The Random Forest is also the most stable of
the three (+/- 0.045 on ROC AUC against the rule's +/- 0.085), which is a second reason
to prefer it beyond the mean.

**What I would report to a reviewer.** Use the Random Forest ranking, but expect
Precision@50 anywhere from roughly 0.69 to 0.98 depending on the client mix, and keep
the hand rule as the fallback — it is close enough that losing the model would not be
a crisis.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [20]:
from sklearn.inspection import permutation_importance

# Refit on one grouped fold so errors can be inspected on unseen clients
tr, te = next(gkf.split(X, y, groups))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X.iloc[tr], y.iloc[tr])
proba = rf.predict_proba(X.iloc[te])[:, 1]

perm = permutation_importance(rf, X.iloc[te], y.iloc[te], n_repeats=5,
                              random_state=42, n_jobs=-1, scoring="roc_auc")
importance = pd.Series(perm.importances_mean, index=model_features).sort_values(ascending=False)
print("Permutation importance, top 10 (drop in ROC AUC when shuffled)")
print(importance.head(10).round(4).to_string())

errors = df.iloc[te].copy()
errors["proba"] = proba
errors["predicted"] = (proba >= 0.5).astype(int)

fp = errors[(errors["predicted"] == 1) & (errors["is_declining"] == 0)]
fn = errors[(errors["predicted"] == 0) & (errors["is_declining"] == 1)]
tp = errors[(errors["predicted"] == 1) & (errors["is_declining"] == 1)]

print(f"\nOn held-out clients: {len(errors):,} pages")
print(f"  true positives : {len(tp):,}")
print(f"  false positives: {len(fp):,}")
print(f"  false negatives: {len(fn):,}\n")

profile_cols = ["imp_feb", "clicks_feb", "ctr_feb", "days_visible_feb",
                "trend_within_feb", "content_age_days", "word_count"]
profile = pd.DataFrame({
    "true positive": tp[profile_cols].median(),
    "false positive": fp[profile_cols].median(),
    "false negative": fn[profile_cols].median(),
}).round(3)
print("Median profile by error type")
print(profile.to_string())

Permutation importance, top 10 (drop in ROC AUC when shuffled)
trend_within_feb      0.0525
imp_per_active_day    0.0233
word_count            0.0154
imp_feb               0.0134
ctr_feb               0.0134
clicks_feb            0.0071
content_age_days      0.0041
days_visible_feb      0.0033
no_h1_impressions     0.0032
word_count_missing    0.0022

On held-out clients: 20,358 pages
  true positives : 300
  false positives: 281
  false negatives: 4,651

Median profile by error type
                  true positive  false positive  false negative
imp_feb                1070.000         395.000         824.000
clicks_feb                1.000           0.000           1.000
ctr_feb                   0.000           0.000           0.001
days_visible_feb         28.000          28.000          28.000
trend_within_feb          0.405           0.537           1.182
content_age_days        212.000         155.000         261.000
word_count                0.000           0.000           0.000

**What the model leans on.** Permutation importance ranks `trend_within_feb` first by
a wide margin (0.0525 drop in ROC AUC when shuffled), then `imp_per_active_day`
(0.0233) and `word_count` (0.0154). Momentum inside February carries roughly twice
the weight of anything else. That matches the single-feature result from my leakage
notebook, where `trend_within_feb` also scored highest alone.

**The error counts change the story the metric table told.** On 20,358 held-out pages
the model produces 300 true positives, 281 false positives, and 4,651 false negatives.
Recall is about 6%: of the pages that actually declined, the model finds one in
sixteen. Precision@50 of 0.836 is still true — the top of the ranking is reliable —
but it measures only 50 rows. The two numbers describe different things, and reporting
only the first would overstate what this model does.

**What the misses look like.** Comparing medians by error type:

| | true positive | false positive | false negative |
|---|---|---|---|
| imp_feb | 1,070 | 395 | 824 |
| trend_within_feb | 0.405 | 0.537 | 1.182 |
| content_age_days | 212 | 155 | 261 |

The false negatives have `trend_within_feb` of 1.182 — they were *growing* through
February and then declined in March. The model has no way to see that coming: every
feature it holds says the page was healthy. The true positives, by contrast, were
already falling (0.405) before the label window opened.

So the model is not weak at ranking; it is blind to a specific case. It finds pages
that were already sliding and continued to slide. It cannot find pages that turned.

**Directional implication.** A February-only feature window can only detect decline
that had already started. Catching turns would need a longer history — several months
of prior momentum rather than two halves of one — which the warehouse supports and my
current design does not use. This is a limitation of the window I chose, not of the
model class.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.